In [2]:
import h5py
import numpy as np
import pandas as pd


In [3]:
H5_FILE = '/data/user/tvaneede/muon_tagging/data_processing/output/h5/merged.h5'
E_THRESHOLD = 60e3  # GeV (60 TeV)


def data_base(year):
    level = "level2pass2a" if year <= 2016 else "level2"
    return f"/data/exp/IceCube/{year}/filtered/{level}"


def good_run_info_path(year):
    if year == 2010:
        return f"{data_base(year)}/IC79_{year}_GoodRunInfo.txt"
    else:
        return f"{data_base(year)}/IC86_{year}_GoodRunInfo.txt"


run_to_season = {}
for year in range(2010, 2023):
    path = good_run_info_path(year)
    season = "IC79_2010" if year == 2010 else f"IC86_{year}"
    try:
        with open(path) as fh:
            for line in fh:
                line = line.strip()
                if not line or line.startswith("RunNum") or line.startswith("("):
                    continue
                parts = line.split()
                if len(parts) < 2:
                    continue
                try:
                    run_to_season[int(parts[0])] = season
                except ValueError:
                    pass
    except FileNotFoundError:
        pass

print(f"Season mapping covers {len(run_to_season)} runs")


Season mapping covers 16982 runs


In [7]:
KEYS = [
    "RecoETot",
    "RecoZenith",
    "FinalTopology",
    "HESEMillipedeFit_PPB0DepositedEnergy",
    "HESEMillipedeFit_PPB0TruncatedDepositedEnergy",
]

frames = {}
with h5py.File(H5_FILE, "r") as f:
    for key in KEYS:
        ds = f[key][:]
        frames[key] = pd.DataFrame({
            "Run":   ds["Run"],
            "Event": ds["Event"],
            key:     ds["value"],
        })

df = frames["RecoETot"].copy()
for key in KEYS[1:]:
    df = df.merge(frames[key], on=["Run", "Event"], how="left")

df["Dataset"] = df["Run"].map(run_to_season).fillna("unknown")

print(f"Total events loaded: {len(df)}")


Total events loaded: 30


In [8]:
df_above = (
    df[df["RecoETot"] > E_THRESHOLD]
    .sort_values("RecoETot", ascending=False)
    .reset_index(drop=True)
)
df_above.index += 1  # 1-based
df_above["RecoCosZenith"] = np.cos(df_above["RecoZenith"])

print(f"{len(df_above)} events above {E_THRESHOLD/1e3:.0f} TeV:")
df_above[[
    "Run", "Event", "Dataset", "RecoETot","RecoCosZenith",
]]


6 events above 60 TeV:


,Run,Event,Dataset,RecoETot,RecoCosZenith
1,136470,35370961,IC86_2021,209546.982731,0.518395
2,136533,50068352,IC86_2021,209010.476221,0.026820
3,131565,3966420,IC86_2018,133582.981695,0.260266
4,122403,49292374,IC86_2013,123372.689804,0.930481
5,129601,56610752,IC86_2017,99345.021898,0.953349
6,131924,42796667,IC86_2018,89202.992288,-0.545368
